To check the number of assets and the end ranking for the set of 8 developing countries

In [3]:
import pandas as pd
import os
from pathlib import Path

def check_phaseout_ranks(base_path, country_codes):
    """
    Check end ranks and total assets phased out for each country and criteria.
    Prints results directly with improved formatting.
    
    Parameters:
    base_path (str): Base path to the data files
    country_codes (list): List of country codes to check
    """
    # Convert base_path to Path object
    base_path = Path(base_path)
    
    # Check if directory exists
    if not base_path.exists():
        print(f"Error: Directory '{base_path}' does not exist!")
        return
    
    # Use the correct filenames pattern from your code
    files = {
        'maturity': 'v2_power_plant_phaseout_order_by_maturity_{}_2050.csv',
        'emission_factor': 'v2_power_plant_phaseout_order_by_emission_factor_{}_2050.csv',
        'benefits_cost_maturity': 'v2_power_plant_phaseout_order_by_emissions_per_OC_maturity_{}_2050.csv'
    }
    
    # Improved table header with better spacing and central alignment for numbers
    print("\n{:<6} {:<22} {:^12} {:^12} {:^12} {:^12} {:^12}".format(
        "Country", "Criteria", "Total Assets", "Coal Assets", "Oil Assets", "Gas Assets", "End Rank"))
    print("-" * 85)  # Adjusted to match the new formatting width
    
    # Track files found
    files_found = 0
    files_checked = 0
    
    for country in country_codes:
        for criterion, file_pattern in files.items():
            filename = file_pattern.format(country)
            file_path = base_path / filename
            files_checked += 1
            
            if not file_path.exists():
                print(f"{country:<6} {criterion:<22} {'File not found':<70}")
                continue
            
            files_found += 1
            try:
                # Read the data
                df = pd.read_csv(file_path)
                
                # Calculate unique assets
                total_assets = df['uniqueforwardassetid'].nunique()
                
                # Calculate assets by subsector
                coal_assets = df[df['subsector'] == 'Coal']['uniqueforwardassetid'].nunique()
                oil_assets = df[df['subsector'] == 'Oil']['uniqueforwardassetid'].nunique()
                gas_assets = df[df['subsector'] == 'Gas']['uniqueforwardassetid'].nunique()
                
                # Get maximum rank (if already calculated)
                if 'rank' in df.columns:
                    max_rank = df['rank'].max()
                else:
                    # If rank is not calculated, calculate it now
                    fuel_order = ['Coal', 'Oil', 'Gas']
                    plant_ranks = {}
                    current_rank = 1
                    
                    # For each year in chronological order
                    for year in sorted(df['year'].unique()):
                        year_df = df[df['year'] == year]
                        
                        # For each fuel type in the specified order
                        for fuel in fuel_order:
                            # Get plants of this fuel type for this year
                            fuel_plants = year_df[year_df['subsector'] == fuel]
                            
                            # Assign ranks to these plants (in CSV order)
                            for _, plant in fuel_plants.iterrows():
                                asset_id = plant['uniqueforwardassetid']
                                if asset_id not in plant_ranks:
                                    plant_ranks[asset_id] = current_rank
                                    current_rank += 1
                    
                    max_rank = max(plant_ranks.values()) if plant_ranks else 0
                
                # Print results with improved formatting - center-aligned numbers
                print("{:<6} {:<22} {:^12} {:^12} {:^12} {:^12} {:^12}".format(
                    country, criterion, total_assets, coal_assets, oil_assets, gas_assets, max_rank))
            
            except Exception as e:
                print(f"{country:<6} {criterion:<22} Error: {str(e)[:60]}")
    
    print("\nPhase-out summary complete.")
    print(f"Files found: {files_found} out of {files_checked} checked")

# Your existing path
base_path = "/Users/yukthabhadane/Documents/Climate Finance Thesis/Paper Alissa Jan 2025/Phase out data"

# List of country codes to check
countries = ['IN', 'ID', 'ZA', 'MX', 'VN', 'IR', 'TH', 'EG']

# Run the check
check_phaseout_ranks(base_path, countries)


Country Criteria               Total Assets Coal Assets   Oil Assets   Gas Assets    End Rank  
-------------------------------------------------------------------------------------
IN     maturity                   927          897           2            30          927     
IN     emission_factor            916          839           3            76          916     
IN     benefits_cost_maturity     925          868           3            56          925     
ID     maturity                   336          308           4            26          336     
ID     emission_factor            339          254           8            79          339     
ID     benefits_cost_maturity     342          300           2            42          342     
ZA     maturity                   110          106           5            1           110     
ZA     emission_factor            111           90           15           8           111     
ZA     benefits_cost_maturity     110          106       